# 第三轮讨论：架构设计深入

## 讨论背景

前两轮讨论已收敛的共识：
- 核心方法：Per-finger RMA + Graph Attention
- 应用靶点：手型 × 物体双向泛化的 in-hand rotation
- Token 粒度：joint-level（可与 per-finger adaptation 解耦）
- Hand-side 机制：dynamic state × static embodiment 的 cross-attention
- Object-side：per-finger local latent → aggregation → global memory
- Local 主路 + Global 残差

本轮聚焦：**架构设计的具体细节**

## 3.1 架构拍板顺序：先定 memory 形态，再定交互链路

**用户反馈**：前两轮已接受 `joint-level token + per-finger local latent + global memory + hand-side cross-attention` 的总体骨架；本轮希望聚焦能直接决定编码实现的架构细节，并重点考虑 20Hz+ 推理约束与 joint-space action 硬约束。

**分析**：

当前 5 个悬留问题并不是彼此独立的。若按“最少返工”的角度看，建议按下面顺序拍板：

| 优先级 | 要先决定的点 | 原因 | 对后续影响 |
|---|---|---|---|
| 1 | `object memory` 是 1 个 token 还是多个 memory tokens | 这直接决定 cross-attention 到底只是全局门控，还是能做多接触区域的信息路由 | 会影响 supervision、finger summary 是否必要、以及时序编码应输出给谁 |
| 2 | 时序 history 放图外还是图内 | 20Hz+ 约束下，这决定整网复杂度上限 | 会影响 local latent 生成方式和 hand/object 交互带宽 |
| 3 | local/global 的监督拆分 | 决定 per-finger latent 是否真的学到“局部交互”，还是退化成重复版全局 latent | 会影响实验可解释性 |
| 4 | 是否显式加 finger summary layer | 主要影响未来多手型扩展与结构可解释性 | 对单手型 MVP 不是第一优先 |
| 5 | cross-attention 具体放哪几层 | 本质上依赖前面几项，因为 memory/token 形态不同，最优交互链路也会变 | 是“后置拍板项” |

从目前约束看，我更倾向把设计空间先压缩成两类 object-side 方案：

1. **单 memory token（HORA-plus 版本）**
   - 形式：`{z_f} -> aggregate -> z_g`，再由 joint tokens 读取 `z_g`
   - 优点：最快、最稳、实现最简；更像“全局协调残差”
   - 风险：cross-attention 容易退化成高级版 gating，难体现“多局部接触路由”的结构优势

2. **少量 memory tokens / slots（推荐作为更有 paper 味的候选）**
   - 形式：`{z_f} -> K 个 object memory slots`，joint/finger summary 再去 cross-attend
   - 优点：能表达“多个局部接触假设 / 多区域交互摘要”，更匹配不对称物体与接触角色切换
   - 风险：如果 K 太大就会逼近重型 object graph；需要控制到很小（如 2~4 个）才能守住 20Hz+

若以 **第一版可编码 + 20Hz+ + 仍保留结构新意** 为目标，一个很像样的折中是：
- **history encoder 放图外**：每指一个小 TCN/GRU，输出 `z_f`
- **object side 用少量固定上限 slots（如 2 或 4）**，而不是单一 `z_g`
- **hand 内部仍用 self-attention + graph bias**
- **hand -> object / object -> hand 才用 cross-attention**

这会让整套东西更像：
`finger-local temporal inference -> compact object memory -> embodiment-aware hand graph -> per-joint action`

而不是把所有问题都丢给一个大 transformer。

**小结**：如果本轮要收敛到“可以开始编码”的层面，第一拍最好先落在 **object memory 的形态与数量上**；这会把后面 4 个问题一起缩到更小的搜索空间。

**待确认**：第一版 MVP，你更倾向 `1 个全局 memory token`，还是 `少量 memory slots（2~4）`？

## 3.2 补读必读论文与代码后的结构约束

**用户反馈**：在继续拍板 object memory 之前，先补读 `AnyMani/source/anymani/papers/context.md` 中列出的必读论文与代码，以免第三轮架构讨论脱离证据。

**分析**：

本轮已补读：
- 论文：HORA 2022、RotateIt 2023、MAGCLA 2023、AnyRotate 2024、GET-Zero 2024、DexNDM 2025、T(R,O) Grasp 2025
- 代码：`TRO-Grasp/model/tro_graph.py`、`TRO-Grasp/model/denoiser.py`、`get_zero/get_zero/rl/models/embodiment_transformer.py`、`get_zero/get_zero/distill/models/embodiment_attention.py`、`get_zero/get_zero/distill/models/embodiment_transformer.py`、`get_zero/get_zero/distill/models/vis_embodiment_transformer.py`

补读后的结论不是“推翻前两轮”，而是把第三轮的搜索空间进一步压缩了：

### 一、关于时序处理：**图外 history encoder 更像默认优选**

1. **HORA** 的 adaptation 模块明确是图外时序编码：先用 proprio/action history 预测 extrinsics，再喂给控制策略。
2. **AnyRotate** 的 student 也是 `history -> latent -> policy` 结构，而不是把长时间序列扔进整个图网络里。
3. **HORA 附录** 还给了一个很关键的负面证据：
   - DR baseline 里直接 concat 长 history 的 MLP 很难训；
   - LSTM 也不理想；
   - TCN 风格更稳。
4. **RotateIt** 虽然用了 transformer 做时序融合，但它面对的是 vision+touch+proprio 多模态序列，而不是我们当前更轻的 proprio/contact 主设定。

**结构含义**：如果我们目标是 `20Hz+` 且第一版仍以 proprio / light tactile 为主，那么更像样的默认方案是：
- 每根手指单独一个小 TCN/GRU 编码 history，输出 `z_f`
- hand graph 只处理“当前时刻的 joint tokens + object memory”
- 不做时空一锅炖的大 transformer

### 二、关于 hand-side：**GET-Zero 的真正贡献点依然是 joint token + graph bias**

从论文和源码都能看清：
- token 单位是 **joint / DoF**，不是 finger；
- 图归纳偏置是加在 attention score 上的 **SPD + parent/child + edge bias**；
- 它在 hand-generalization 上最关键的是：
  1. token 粒度足够细；
  2. embodiment 信息不是简单 concat，而是通过图结构/偏置参与注意力。

**结构含义**：
- 我们把 token 粒度定在 joint-level 是对的；
- hand 内部的默认通信应该仍是 **self-attention + graph bias**；
- 若要加 cross-attention，应该加在 **dynamic joint stream ↔ static embodiment stream**，以及 **hand ↔ object memory** 两类跨模态链路，而不是替代整套 hand graph。

### 三、关于 object-side：**T(R,O) 最值得借的是“显式 hand-object relation”，不是 patch graph 本体**

从 `tro_graph.py` / `denoiser.py` 可见：
- 物侧是多 patch token；
- 手侧是 link nodes；
- OR / RR 两类边都带显式 SE(3) 关系；
- 整套系统是为 diffusion grasp synthesis 设计的，在线闭环 policy 太重。

**结构含义**：
- 我们不该把 object side 做成完整 patch graph；
- 但也不该退化成完全无结构的纯 `[CLS]`；
- 更合理的折中是：
  - 让 `z_f` 聚合出 **少量 object memory slots**；
  - 这些 slots 作为 hand 读取的 object-side memory；
  - 若以后要加显式 hand-object relation，优先加在 distal joints / fingertips 到 memory 的交互上，而不是引入大量 object patches。

### 四、关于 local / global supervision：**文献支持“local 学 interaction，global 学 object-level state”**

1. **HORA / RotateIt** 的 global latent 本质上学的是 object/extrinsics；
2. **AnyRotate** 明确说明 rich tactile 的价值在于检测局部接触不稳定、恢复 grasp；
3. **DexNDM** 强调 factorization 的价值在于把预测目标局部化，从而过滤与本局部无关的高维噪声。

**结构含义**：
- `z_f` 不应该都去回归同一个完整 object state；
- 更合理的是：
  - **local loss**：预测本指未来 contact / contact pose / force proxy / slip tendency
  - **global loss**：预测 privileged object extrinsics / object pose / angular velocity / coarse geometry code

### 五、关于 finger summary layer：**单手 MVP 不是硬必需，但多手型故事里价值明显上升**

1. **MAGCLA** 说明 finger-level cooperation 确实是可解释层；
2. **GET-Zero** 证明只用 joint layer 也能学，但它的泛化对象主要还是 LEAP family；
3. 如果未来要把 “不同 joint 数量 / 不同 finger layout” 对齐起来，显式 finger summary 会是很自然的中间语义层。

**结构含义**：
- 单手型 MVP：可先不上 finger summary，减少实现负担；
- 若第三轮目标是直接把 future multi-hand 接口也设计好：可以考虑保留一个 **可选的 finger pooling / summary hook**，但不一定第一版就启用复杂的双向 decode。

### 六、当前最像样的架构收缩版本

综合论文与代码证据，第三轮里最稳、也最像“能开始编码”的骨架更像：

1. **Per-finger temporal encoder（图外）**：小 TCN/GRU，从本指 history 得到 `z_f`
2. **Object-side memory**：由 `{z_f}` 聚合出 `K` 个小 memory slots（推荐先考虑 `K=2~4`）
3. **Dynamic joint tokens**：当前 joint states / previous actions / local obs
4. **Static embodiment tokens**：joint limits / rest pose / link geometry / graph metadata
5. **Cross-Attn 0**：dynamic joint tokens 查询 static embodiment tokens
6. **Hand graph encoder**：self-attention + graph bias（SPD / parent-child / edge bias）
7. **Cross-Attn 1**：hand tokens 读取 object memory slots
8. **Action head**：joint-space output，且保持 `local 主路 + global/object residual`

也就是更接近：
`finger-local history inference -> compact object memory -> embodiment-aware joint graph -> joint action`

而不是：
`all signals -> one huge transformer -> action`

**小结**：补读论文和代码后，第三轮最值得优先拍板的，不再是“要不要 graph / 要不要 RMA”这种大方向，而是三个实现层级问题：
1. object memory 用 `1 / 2 / 4` 个 slots？
2. finger summary layer 第一版上不上？
3. cross-attention 精确落在 `dynamic↔static` 和 `hand↔object` 的哪一处？

**待确认**：你下一步更想先收敛哪一项：`object memory 数量`、`finger summary layer`，还是 `cross-attention 的具体拓扑`？

## 3.3 回到可编码层面的最小分歧

**用户反馈**：希望讨论语气更正常、直接，不要过度渲染。

**分析**：

在补读论文和代码之后，当前对架构最关键的判断可以收敛成下面三条：

1. **时序模块优先放图外**
   - 对当前 `20Hz+` 约束更稳；
   - 与 HORA / AnyRotate 的训练形态更一致；
   - 也更容易把 `per-finger latent` 和 `joint-level graph` 解耦。

2. **hand 内部默认仍用 self-attention + graph bias**
   - 这是 GET-Zero 最有价值的部分；
   - cross-attention 更适合放在跨模态交互，而不是取代 hand graph 本身。

3. **接下来最值得拍板的是 object-side memory 的最小形态**
   - 如果只有 1 个 global token，那么 hand->object 的 cross-attention 很容易退化成门控；
   - 如果一开始就上完整 patch graph，又明显超出第一版在线策略的复杂度预算；
   - 因此更现实的分歧是：`1 个 token` 还是 `少量 slots（2~4）`。

在这个前提下，当前最接近“可开始编码”的最小骨架是：
- 每指一个 history encoder，输出 `z_f`
- `{z_f}` 聚合成 object-side memory
- dynamic joint tokens 查询 static embodiment tokens
- hand graph 用 graph bias 做 joint-level message passing
- hand tokens 再读取 object-side memory
- action 仍然输出 joint-space，并保留 local 主路 + global 残差

**小结**：大的方向已经基本清楚，剩下最影响实现的是 `object memory` 以及它对应的 `cross-attention` 形态。

**待确认**：下一步是先把 object memory 定成单 token 还是少量 slots？

## 3.4 交叉注意力（Cross-Attention）在当前架构里的作用

**用户反馈**：希望后续讨论采用更清楚、讲原理、分点说明的表达方式，并继续聚焦 cross-attention 的具体作用与放置位置。

**分析**：

### 一、交叉注意力在本问题里的定义

在当前讨论中，我们有两类不同来源的序列：

1. **Hand-side tokens**
   - joint-level dynamic tokens：来自当前关节状态、上一时刻动作、局部观测
   - 或进一步得到的 finger/joint summary tokens

2. **Condition-side tokens**
   - static embodiment tokens：URDF、joint limit、rest pose、link geometry、graph meta-data
   - object-side memory tokens：由 per-finger history encoder 输出的 `{z_f}` 聚合而来

如果记 hand-side 序列为 $X \in \mathbb{R}^{L_x 	imes d}$，condition-side 序列为 $Y \in \mathbb{R}^{L_y 	imes d}$，
则交叉注意力可以写成：

$$
Q = X W_Q, \quad K = Y W_K, \quad V = Y W_V
$$

$$
\mathrm{CrossAttn}(X, Y)=\mathrm{softmax}\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

其中输出长度由 $X$ 决定，也就是：
- **谁提供 $Q$，谁就决定“我要更新谁”**；
- **谁提供 $K,V$，谁就决定“我能从哪里取信息”**。

### 二、为什么这里不能把所有交互都写成 self-attention

如果把所有 token（joint、embodiment、object memory）直接拼成一个大序列做 self-attention，当然在数学上可行；但从结构上会有三个问题：

1. **语义混合过早**
   - joint state 是动态控制量；
   - embodiment token 是静态结构先验；
   - object memory 是交互记忆。
   - 三者统计性质不同，直接混在一起会增加 disentangle 的难度。

2. **可解释性变差**
   - 很难明确回答“当前 joint token 到底主要在查询 hand structure，还是在查询 object state”。

3. **复杂度与调试成本更高**
   - 对第一版可编码架构不利。

因此，在当前问题里，把 cross-attention 作为**跨模态信息注入模块**，而把 self-attention + graph bias 作为**hand 内部结构建模模块**，会更清楚。

### 三、当前最自然的两处 cross-attention

#### 1. Dynamic joint stream × static embodiment stream

这里的目标是：
> 当前 joint 的数值状态，应该如何在“这只手的结构”里被解释？

设：
- dynamic joint tokens 为 $X_{dyn}$
- static embodiment tokens 为 $Y_{emb}$

则可写为：

$$
H_{emb} = \mathrm{CrossAttn}(X_{dyn}, Y_{emb})
$$

其含义是：
- $Q$ 来自 dynamic joint tokens，表示“当前关节状态提出的问题”；
- $K,V$ 来自 embodiment tokens，表示“这只手的结构知识库”；
- 输出仍是 joint-length 序列，因此后续 action head 仍能保持 joint-space 对齐。

**优点**：
- 符合 hand-generalization 的核心需求；
- 比简单 concat 更容易解释；
- 不破坏 joint-level tokenization。

#### 2. Hand-side tokens × object-side memory

这里的目标是：
> 当前 hand token 需要从 object interaction memory 中读取什么上下文，来修正控制？

设：
- hand tokens 为 $X_{hand}$
- object memory 为 $Y_{obj}$

则可写为：

$$
H_{obj} = \mathrm{CrossAttn}(X_{hand}, Y_{obj})
$$

其含义是：
- hand token 作为 query；
- object memory 作为 key/value；
- 输出长度依然与 hand token 一致，所以适合直接接 action head 或 residual head。

### 四、哪些地方更适合 self-attention，而不是 cross-attention

#### 1. hand 内部 joint-to-joint 通信

这部分更适合：

$$
\mathrm{SelfAttn}(X_{hand}) + \mathrm{GraphBias}
$$

原因是：
- joint 之间原本就处于同一结构系统内；
- GET-Zero 已经证明 graph bias 对这种通信是有效的；
- 这里的核心问题不是“从外部记忆检索信息”，而是“在关节图内部传播信息”。

#### 2. `{z_f}` 到 object memory 的聚合

这里不一定必须用 cross-attention。

如果 `{z_f}` 只是聚合成 1 个 global token，那么：
- mean pooling
- attention pooling
- small DeepSets aggregator

都可能足够。

只有当 object side 明确采用多个 memory slots 时，才更有必要用：
- learned slots 作为 query
- `{z_f}` 作为 key/value

即：

$$
M_{obj} = \mathrm{CrossAttn}(X_{slot}, \{z_f\})
$$

此时 cross-attention 的意义才比较明确：
- 每个 slot 主动从所有 finger local latent 中“分工读取”不同信息。

### 五、从当前证据看，第一版最合理的 cross-attention 拓扑

如果目标是：
- joint-space action
- 20Hz+
- 单手 MVP 先跑通
- 未来保留多手型扩展接口

那么当前最合理的顺序是：

1. **per-finger history encoder（图外）**
   $$
   z_f = \phi_f(h_f)
   $$

2. **`{z_f}` -> object memory**
   - 若 memory 很小，可先用 pooling；
   - 若用 2~4 个 slots，可用 slot-to-finger cross-attention。

3. **dynamic joint tokens 查询 static embodiment tokens**
   $$
   X_{emb} = \mathrm{CrossAttn}(X_{dyn}, Y_{emb})
   $$

4. **hand graph encoder 做内部通信**
   $$
   X_{hand} = \mathrm{SelfAttn}(X_{emb}) + \mathrm{GraphBias}
   $$

5. **hand tokens 查询 object memory**
   $$
   X_{out} = \mathrm{CrossAttn}(X_{hand}, Y_{obj})
   $$

6. **joint-space action head**
   $$
   a_j = \pi_{local}(x_j, z_{f(j)}, e_j) + \Delta \pi_{global}(x_j, Y_{obj})
   $$

### 六、一个关键判断

当前最影响 cross-attention 形态的，不是“要不要用 cross-attention”，而是：

> **object-side memory 到底是 1 个 token，还是多个 slots？**

因为：
- 如果只有 **1 个 token**，hand->object 的 cross-attention 很容易退化成一种全局 gating；
- 如果有 **2~4 个 slots**，hand token 才真正有“按需检索不同 object context”的空间。

**小结**：
- hand 内部：优先 `self-attention + graph bias`
- hand 与 embodiment / object memory：优先 `cross-attention`
- object-side 是否采用多个 memory slots，将直接决定 cross-attention 是否只是门控，还是具备真实的信息路由能力。

**待确认**：如果我们现在专门讨论 cross-attention 的具体拓扑，那么下一步最关键的是先决定：`object-side memory` 用单 token，还是用少量 slots？